# Assignment 4 — Regularization and Dropout | Raisin Dataset
**Team notebook — Colab-ready.** Adapted from the professor’s `Regularization.ipynb` and the team extension. The original NumPy FC network, L1/L2 gradients, dropout, multi-seed sweeps and augmentation are reused. **No metrics are fabricated:** run all cells to produce results.

Dataset: UCI Raisin (ID 850), binary classification Kecimen/Besni. The test set is held out until the final evaluation.

## 0. Google Drive setup
1. Upload the complete `Assignment4_Raisin` folder (including `scripts`) to **My Drive**.
2. Open `Assignment_4_Raisin_Colab.ipynb` in Colab and run this cell. You may edit `DRIVE_FOLDER` if stored elsewhere.
3. Alternatively, upload the companion ZIP to Colab and unzip it under `/content/Assignment4_Raisin`.

The notebook imports the original helpers from Drive. The UCI dataset is downloaded automatically and cached to the project folder.

In [ ]:
from pathlib import Path
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = Path('/content/drive/MyDrive/Assignment4_Raisin')
else:
    DRIVE_FOLDER = Path.cwd()
PROJECT = DRIVE_FOLDER if (DRIVE_FOLDER / 'scripts' / 'reg_utils.py').exists() else Path('/content/Assignment4_Raisin')
assert (PROJECT / 'scripts' / 'reg_utils.py').exists(), f'Upload the complete folder first: {PROJECT}'
sys.path.insert(0, str(PROJECT / 'scripts'))
(PROJECT / 'results').mkdir(exist_ok=True)
print('Project:', PROJECT)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from reg_utils import (sigmoid, relu, compute_cost, predict, forward_propagation,
                       backward_propagation, update_parameters, initialize_parameters)
from testCases import *
from public_tests import *
# Legacy helper compatibility with recent NumPy versions
if not hasattr(np, 'int'): np.int = int
plt.rcParams['figure.figsize'] = (9, 4)
print('Imports OK')

## 1. Dataset and leakage-free preprocessing
Fetch UCI Raisin from the original source (ID 850). Split 60/20/20, stratified. Fit the scaler **only on training data**. Network arrays use `(features, samples)` and labels `(1, samples)`.

In [ ]:
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    %pip -q install ucimlrepo
    from ucimlrepo import fetch_ucirepo
cache = PROJECT / 'raisin_data.csv'
if cache.exists():
    df = pd.read_csv(cache)
else:
    raisin = fetch_ucirepo(id=850)
    df = raisin.data.features.copy()
    df['Class'] = raisin.data.targets.iloc[:, 0].to_numpy()
    df.to_csv(cache, index=False)
print(df.shape, df['Class'].value_counts().to_dict())
display(df.head())

In [ ]:
feature_names = [c for c in df.columns if c != 'Class']
labels = sorted(df['Class'].unique())
label_map = {labels[0]: 0, labels[1]: 1}
X_all = df[feature_names].astype(float).to_numpy()
y_all = df['Class'].map(label_map).astype(int).to_numpy()
X_tmp, X_test_raw, y_tmp, y_test_raw = train_test_split(X_all, y_all, test_size=.20, stratify=y_all, random_state=42)
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(X_tmp, y_tmp, test_size=.25, stratify=y_tmp, random_state=42)
scaler = StandardScaler().fit(X_train_raw)
train_X, val_X, test_X = [scaler.transform(x).T for x in (X_train_raw, X_val_raw, X_test_raw)]
train_Y, val_Y, test_Y = [y.reshape(1, -1) for y in (y_train_raw, y_val_raw, y_test_raw)]
print('Train/validation/test:', train_X.shape[1], val_X.shape[1], test_X.shape[1], 'features:', train_X.shape[0]); print('Label mapping:',label_map)
assert train_X.shape == (7,540) and val_X.shape == (7,180) and test_X.shape == (7,180)

In [ ]:
pca = PCA(n_components=2).fit(train_X.T)
for X, Y, name in [(train_X,train_Y,'Train'),(val_X,val_Y,'Validation'),(test_X,test_Y,'Test')]:
    xy = pca.transform(X.T)
    plt.scatter(xy[:,0],xy[:,1],c=Y.ravel(),alpha=.5,s=14,label=name)
plt.xlabel('PC1');plt.ylabel('PC2');plt.title('Raisin: 2D PCA projection (visualization only)');plt.legend();plt.show()

## 2. Original professor functions — L2 and dropout (reused)

In [ ]:
# GRADED FUNCTION: compute_cost_with_regularization

def compute_cost_with_regularization(A3, Y, parameters, lambd):
    """
    Implement the cost function with L2 regularization. See formula (2) above.

    Arguments:
    A3 -- post-activation, output of forward propagation, of shape (output size, number of examples)
    Y -- "true" labels vector, of shape (output size, number of examples)
    parameters -- python dictionary containing parameters of the model

    Returns:
    cost - value of the regularized loss function (formula (2))
    """
    m = Y.shape[1]
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    W3 = parameters["W3"]

    cross_entropy_cost = compute_cost(A3, Y) # This gives you the cross-entropy part of the cost

    #(≈ 1 lines of code)
    # L2_regularization_cost =
    # YOUR CODE STARTS HERE
    L2_regularization_cost = (np.sum(np.square(W1))+np.sum(np.square(W2))+np.sum(np.square(W3)))*(lambd/(2*m))

    # YOUR CODE ENDS HERE

    cost = cross_entropy_cost + L2_regularization_cost

    return cost

In [ ]:
# GRADED FUNCTION: backward_propagation_with_regularization

def backward_propagation_with_regularization(X, Y, cache, lambd):
    """
    Implements the backward propagation of our baseline model to which we added an L2 regularization.

    Arguments:
    X -- input dataset, of shape (input size, number of examples)
    Y -- "true" labels vector, of shape (output size, number of examples)
    cache -- cache output from forward_propagation()
    lambd -- regularization hyperparameter, scalar

    Returns:
    gradients -- A dictionary with the gradients with respect to each parameter, activation and pre-activation variables
    """

    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache

    dZ3 = A3 - Y
    #(≈ 1 lines of code)
    # dW3 = 1./m * np.dot(dZ3, A2.T) + None
    # YOUR CODE STARTS HERE
    dW3 = 1./m * np.dot(dZ3, A2.T) + lambd/m*W3

    # YOUR CODE ENDS HERE
    db3 = 1. / m * np.sum(dZ3, axis=1, keepdims=True)

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    #(≈ 1 lines of code)
    # dW2 = 1./m * np.dot(dZ2, A1.T) + None
    # YOUR CODE STARTS HERE
    dW2 = 1./m * np.dot(dZ2, A1.T) + lambd/m*W2

    # YOUR CODE ENDS HERE
    db2 = 1. / m * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    #(≈ 1 lines of code)
    # dW1 = 1./m * np.dot(dZ1, X.T) + None
    # YOUR CODE STARTS HERE
    dW1 = 1./m * np.dot(dZ1, X.T) + lambd/m*W1

    # YOUR CODE ENDS HERE
    db1 = 1. / m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1,
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}

    return gradients

In [ ]:
# GRADED FUNCTION: forward_propagation_with_dropout

def forward_propagation_with_dropout(X, parameters, keep_prob = 0.5):
    """
    Implements the forward propagation: LINEAR -> RELU + DROPOUT -> LINEAR -> RELU + DROPOUT -> LINEAR -> SIGMOID.

    Arguments:
    X -- input dataset, of shape (2, number of examples)
    parameters -- python dictionary containing your parameters "W1", "b1", "W2", "b2", "W3", "b3":
                    W1 -- weight matrix of shape (20, 2)
                    b1 -- bias vector of shape (20, 1)
                    W2 -- weight matrix of shape (3, 20)
                    b2 -- bias vector of shape (3, 1)
                    W3 -- weight matrix of shape (1, 3)
                    b3 -- bias vector of shape (1, 1)
    keep_prob - probability of keeping a neuron active during drop-out, scalar

    Returns:
    A3 -- last activation value, output of the forward propagation, of shape (1,1)
    cache -- tuple, information stored for computing the backward propagation
    """

    np.random.seed(1)

    # retrieve parameters
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    # LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    #(≈ 4 lines of code)         # Steps 1-4 below correspond to the Steps 1-4 described above.
    # D1 =                                           # Step 1: initialize matrix D1 = np.random.rand(..., ...)
    # D1 =                                           # Step 2: convert entries of D1 to 0 or 1 (using keep_prob as the threshold)
    # A1 =                                           # Step 3: shut down some neurons of A1
    # A1 =                                           # Step 4: scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    D1 = np.random.rand(A1.shape[0],A1.shape[1])
    D1 = (D1 < keep_prob).astype(int)
    A1 = A1 * D1
    A1 = A1/keep_prob

    # YOUR CODE ENDS HERE
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    #(≈ 4 lines of code)
    # D2 =                                           # Step 1: initialize matrix D2 = np.random.rand(..., ...)
    # D2 =                                           # Step 2: convert entries of D2 to 0 or 1 (using keep_prob as the threshold)
    # A2 =                                           # Step 3: shut down some neurons of A2
    # A2 =                                           # Step 4: scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    D2 = np.random.rand(A2.shape[0],A2.shape[1])
    D2 = (D2 < keep_prob).astype(int)
    A2 = A2 * D2
    A2 = A2/keep_prob

    # YOUR CODE ENDS HERE
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    cache = (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3)

    return A3, cache

In [ ]:
# GRADED FUNCTION: backward_propagation_with_dropout

def backward_propagation_with_dropout(X, Y, cache, keep_prob):
    """
    Implements the backward propagation of our baseline model to which we added dropout.

    Arguments:
    X -- input dataset, of shape (2, number of examples)
    Y -- "true" labels vector, of shape (output size, number of examples)
    cache -- cache output from forward_propagation_with_dropout()
    keep_prob - probability of keeping a neuron active during drop-out, scalar

    Returns:
    gradients -- A dictionary with the gradients with respect to each parameter, activation and pre-activation variables
    """

    m = X.shape[1]
    (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3) = cache

    dZ3 = A3 - Y
    dW3 = 1./m * np.dot(dZ3, A2.T)
    db3 = 1./m * np.sum(dZ3, axis=1, keepdims=True)
    dA2 = np.dot(W3.T, dZ3)
    #(≈ 2 lines of code)
    # dA2 =                # Step 1: Apply mask D2 to shut down the same neurons as during the forward propagation
    # dA2 =                # Step 2: Scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    dA2 = D2 * dA2
    dA2 = dA2 / keep_prob
    # YOUR CODE ENDS HERE
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1./m * np.dot(dZ2, A1.T)
    db2 = 1./m * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    #(≈ 2 lines of code)
    # dA1 =                # Step 1: Apply mask D1 to shut down the same neurons as during the forward propagation
    # dA1 =                # Step 2: Scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    dA1 = D1 * dA1
    dA1 = dA1 /keep_prob

    # YOUR CODE ENDS HERE
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1./m * np.dot(dZ1, X.T)
    db1 = 1./m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1,
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}

    return gradients

## 3. Team extension — shared training utilities (reused)

In [ ]:
# ============ Utilidades de la Parte 2 ============
import time
import multiprocessing as mp
import pandas as pd
from reg_utils import compute_cost, backward_propagation, forward_propagation, update_parameters, relu, sigmoid

 # val_X and val_Y are the independent validation split defined above
SEEDS = [3, 11, 23]  # Increase to five seeds for the final report              # seed=3 reproduce la inicialización del notebook original


def init_params(layers, seed):
    """Igual que reg_utils.initialize_parameters (que fija seed=3), pero con seed configurable."""
    np.random.seed(seed)
    p = {}
    for l in range(1, len(layers)):
        p["W%d" % l] = np.random.randn(layers[l], layers[l - 1]) / np.sqrt(layers[l - 1])
        p["b%d" % l] = np.zeros((layers[l], 1))
    return p


def cross_entropy(A3, Y, eps=1e-12):
    """Entropía cruzada media con clipping (solo para reportar; evita inf si la salida satura)."""
    A = np.clip(A3, eps, 1 - eps)
    return float(np.mean(-(Y * np.log(A) + (1 - Y) * np.log(1 - A))))


def accuracy(X, Y, params):
    A3, _ = forward_propagation(X, params)      # evaluación SIN dropout
    return float(np.mean((A3 > 0.5) == Y))


def train(X, Y, Xv, Yv, reg="none", lambd=0.0, keep_prob=1.0, lr=0.3, iters=2500, seed=3,
          dropout_mode="stochastic", layers=(20, 3, 1), curve_every=250, X_orig=None, Y_orig=None):
    """
    reg: 'none' | 'l2' | 'l1' | 'dropout'.
    dropout_mode: 'stochastic' (máscara nueva en cada iteración) | 'fixed' (la función del notebook, que reinicia seed(1)).
    X_orig, Y_orig: entrenamiento ORIGINAL; las métricas de entrenamiento se reportan sobre él.
    """
    X_orig = X if X_orig is None else X_orig
    Y_orig = Y if Y_orig is None else Y_orig
    params = init_params([X.shape[0], *layers], seed)
    rng = np.random.RandomState(seed + 1000)
    curve = []
    for i in range(iters):
        if reg == "dropout":
            if dropout_mode == "fixed":
                A3, cache = forward_propagation_with_dropout(X, params, keep_prob)
            else:
                A3, cache = forward_propagation_with_dropout_stochastic(X, params, keep_prob, rng)
            grads = backward_propagation_with_dropout(X, Y, cache, keep_prob)
        else:
            A3, cache = forward_propagation(X, params)
            if reg == "l2":
                grads = backward_propagation_with_regularization(X, Y, cache, lambd)
            elif reg == "l1":
                grads = backward_propagation_with_l1(X, Y, cache, lambd)
            else:
                grads = backward_propagation(X, Y, cache)
        params = update_parameters(params, grads, lr)
        if i % curve_every == 0 or i == iters - 1:
            curve.append((i, accuracy(X_orig, Y_orig, params), accuracy(Xv, Yv, params)))

    W = np.concatenate([params["W%d" % l].ravel() for l in (1, 2, 3)])
    A3t, _ = forward_propagation(X_orig, params)
    A3v, _ = forward_propagation(Xv, params)
    metrics = dict(train_acc=accuracy(X_orig, Y_orig, params), val_acc=accuracy(Xv, Yv, params),
                   train_ce=cross_entropy(A3t, Y_orig), val_ce=cross_entropy(A3v, Yv),
                   w_l1=float(np.abs(W).sum()), w_l2=float(np.sqrt((W ** 2).sum())),
                   frac_small=float(np.mean(np.abs(W) < 1e-2)), curve=curve)
    return params, metrics


def fit(cfg, seed):
    """Entrena una configuración (aplicando aumento de datos si corresponde) y devuelve (parámetros, métricas)."""
    X, Y = train_X, train_Y
    if cfg.get("aug") == "jitter":
        X, Y = augment_jitter(train_X, train_Y, cfg["k"], cfg["sigma"], seed=seed)
    elif cfg.get("aug") == "interp":
        X, Y = augment_interpolate(train_X, train_Y, cfg["k"], seed=seed)
    kw = {k: cfg[k] for k in ("reg", "lambd", "keep_prob", "dropout_mode") if k in cfg}
    with np.errstate(all="ignore"):
        params, m = train(X, Y, val_X, val_Y, seed=seed, X_orig=train_X, Y_orig=train_Y, **kw)
    m.update(label=cfg["label"], cfg=cfg, seed=seed, n_train_used=int(X.shape[1]))
    return params, m


def _job(task):
    cfg, seed = task
    return fit(cfg, seed)[1]


def run_sweep(cfgs, seeds=SEEDS, workers=1):
    """Entrena todas las configuraciones x semillas (en paralelo si el sistema permite 'fork')."""
    tasks = [(c, s) for c in cfgs for s in seeds]
    t0 = time.time()
    if workers > 1 and "fork" in mp.get_all_start_methods():
        with mp.get_context("fork").Pool(workers) as pool:
            res = pool.map(_job, tasks, chunksize=1)
    else:
        res = [_job(t) for t in tasks]
    print("%d entrenamientos en %.0f s" % (len(tasks), time.time() - t0))
    return res


def tabla(results, xkey=None):
    """Resumen por configuración: media y desviación estándar sobre las semillas."""
    rows = []
    for l in dict.fromkeys(r["label"] for r in results):
        rs = [r for r in results if r["label"] == l]
        g = lambda k: np.array([r[k] for r in rs])
        rows.append({"configuración": l, "x": rs[0]["cfg"].get(xkey) if xkey else np.nan,
                     "train_%": 100 * g("train_acc").mean(), "train_std": 100 * g("train_acc").std(),
                     "val_%": 100 * g("val_acc").mean(), "val_std": 100 * g("val_acc").std(),
                     "brecha": 100 * (g("train_acc").mean() - g("val_acc").mean()),
                     "val_CE": g("val_ce").mean(), "|W|_1": g("w_l1").mean(),
                     "pesos<0.01": g("frac_small").mean(), "n_train": rs[0]["n_train_used"]})
    return pd.DataFrame(rows)


def mostrar(df, cols=None):
    cols = cols or ["configuración", "train_%", "train_std", "val_%", "val_std", "brecha", "val_CE", "|W|_1", "pesos<0.01"]
    print(df[cols].round(3).to_string(index=False))


def plot_acc(df, base, xlabel, title, logx=True, ax=None):
    ax = ax or plt.gca()
    ax.errorbar(df.x, df["train_%"], df.train_std, marker="o", capsize=3, label="entrenamiento")
    ax.errorbar(df.x, df["val_%"], df.val_std, marker="s", capsize=3, label="validación")
    ax.axhline(base["val_%"], color="C1", ls="--", lw=1, label="validación sin regularizar")
    ax.axhline(base["train_%"], color="C0", ls="--", lw=1, label="entrenamiento sin regularizar")
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel("precisión (%)"); ax.set_title(title)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)


plt.rcParams["figure.figsize"] = (11, 4)
print("Ready. Seeds:", SEEDS)


## 4. L1 implementation and numerical gradient check (reused)

In [ ]:
def compute_cost_with_l1(A3, Y, parameters, lambd):
    """Costo de entropía cruzada + (lambda/m) * suma de |W| de las 3 capas."""
    m = Y.shape[1]
    cross_entropy_cost = compute_cost(A3, Y)
    L1_regularization_cost = (lambd / m) * (np.sum(np.abs(parameters["W1"])) +
                                            np.sum(np.abs(parameters["W2"])) +
                                            np.sum(np.abs(parameters["W3"])))
    return cross_entropy_cost + L1_regularization_cost


def backward_propagation_with_l1(X, Y, cache, lambd):
    """Retropropagación normal + (lambda/m) * sign(W) en dW1, dW2, dW3."""
    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache
    gradients = backward_propagation(X, Y, cache)                       # gradientes sin regularizar
    gradients["dW3"] = gradients["dW3"] + (lambd / m) * np.sign(W3)
    gradients["dW2"] = gradients["dW2"] + (lambd / m) * np.sign(W2)
    gradients["dW1"] = gradients["dW1"] + (lambd / m) * np.sign(W1)
    return gradients

In [ ]:
def gradient_check(reg, lambd, seed=0):
    cost_fn = compute_cost_with_l1 if reg == "l1" else compute_cost_with_regularization
    grad_fn = backward_propagation_with_l1 if reg == "l1" else backward_propagation_with_regularization
    p = init_params([2, 20, 3, 1], 3)
    for _ in range(50):                                   # unos pasos para salir de la inicialización
        p = update_parameters(p, grad_fn(train_X, train_Y, forward_propagation(train_X, p)[1], lambd), 0.3)
    g = grad_fn(train_X, train_Y, forward_propagation(train_X, p)[1], lambd)
    rng, worst, eps = np.random.RandomState(seed), 0, 1e-6
    for l in (1, 2, 3):
        W = p["W%d" % l]
        for _ in range(8):
            i, j = rng.randint(W.shape[0]), rng.randint(W.shape[1])
            old = W[i, j]
            W[i, j] = old + eps; cp = cost_fn(forward_propagation(train_X, p)[0], train_Y, p, lambd)
            W[i, j] = old - eps; cm = cost_fn(forward_propagation(train_X, p)[0], train_Y, p, lambd)
            W[i, j] = old
            num, ana = (cp - cm) / (2 * eps), g["dW%d" % l][i, j]
            worst = max(worst, abs(num - ana) / max(1e-8, abs(num) + abs(ana)))
    return worst

for reg, lam in [("l2", 0.7), ("l1", 0.7), ("l1", 3.0)]:
    print("gradient check %s, lambda=%-4s -> error relativo máximo = %.2e" % (reg, lam, gradient_check(reg, lam)))

## 5. Stochastic dropout implementation (reused)

In [ ]:
def forward_propagation_with_dropout_stochastic(X, parameters, keep_prob, rng):
    """Igual que forward_propagation_with_dropout, pero SIN reiniciar la semilla: máscara nueva en cada llamada."""
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    W3, b3 = parameters["W3"], parameters["b3"]
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    D1 = (rng.rand(*A1.shape) < keep_prob).astype(int)      # pasos 1-2: máscara de 0/1
    A1 = A1 * D1 / keep_prob                                # pasos 3-4: apagar y reescalar
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    D2 = (rng.rand(*A2.shape) < keep_prob).astype(int)
    A2 = A2 * D2 / keep_prob
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)
    return A3, (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3)

## 6. Data augmentation — train only (reused)

In [ ]:
def augment_jitter(X, Y, k, sigma_rel, seed=0):
    """k copias de cada punto con ruido gaussiano de desviación sigma_rel * std(característica); misma etiqueta."""
    rng = np.random.RandomState(seed)
    std = X.std(axis=1, keepdims=True)
    Xs, Ys = [X], [Y]
    for _ in range(k):
        Xs.append(X + rng.randn(*X.shape) * sigma_rel * std)
        Ys.append(Y.copy())
    return np.concatenate(Xs, axis=1), np.concatenate(Ys, axis=1)


def augment_interpolate(X, Y, k, n_neighbors=5, seed=0):
    """Estilo SMOTE: k puntos nuevos por punto original, interpolando con un vecino cercano de la misma clase."""
    rng = np.random.RandomState(seed)
    Xs, Ys = [X], [Y]
    for c in np.unique(Y):
        idx = np.where(Y[0] == c)[0]
        Xc = X[:, idx]
        d = ((Xc[:, :, None] - Xc[:, None, :]) ** 2).sum(axis=0)
        np.fill_diagonal(d, np.inf)
        nn = np.argsort(d, axis=1)[:, :n_neighbors]
        for _ in range(k):
            j = nn[np.arange(len(idx)), rng.randint(0, n_neighbors, len(idx))]
            lam = rng.rand(1, len(idx))
            Xs.append(Xc * lam + Xc[:, j] * (1 - lam))
            Ys.append(np.full((1, len(idx)), c, dtype=Y.dtype))
    return np.concatenate(Xs, axis=1), np.concatenate(Ys, axis=1)




## 7. Baseline and L2 sweep
Initial grids are compact to keep Colab runtime reasonable. Extend λ values and seeds for a more exhaustive study. All methods share the same split and architecture.

In [ ]:
cfg_base = [dict(label='Baseline',reg='none')]
cfg_l2 = [dict(label=f'L2 λ={lam}',reg='l2',lambd=lam) for lam in (.001,.01,.1,1.,10.)]
res_base = run_sweep(cfg_base); res_l2 = run_sweep(cfg_l2)
df_base = tabla(res_base).iloc[0];df_l2=tabla(res_l2,'lambd')
mostrar(pd.concat([tabla(res_base),df_l2],ignore_index=True))
fig,ax=plt.subplots();plot_acc(df_l2,df_base,'λ (L2)','L2 regularization',ax=ax);plt.show()

## 8. L1 sweep

In [ ]:
cfg_l1=[dict(label=f'L1 λ={lam}',reg='l1',lambd=lam) for lam in (.001,.01,.1,1.,10.)]
res_l1=run_sweep(cfg_l1);df_l1=tabla(res_l1,'lambd');mostrar(df_l1)
fig,ax=plt.subplots();plot_acc(df_l1,df_base,'λ (L1)','L1 regularization',ax=ax);plt.show()

## 9. Dropout sweep — fresh random masks each iteration

In [ ]:
cfg_dp=[dict(label=f'Dropout keep={kp}',reg='dropout',keep_prob=kp,dropout_mode='stochastic') for kp in (.95,.9,.8,.7,.5)]
res_dp=run_sweep(cfg_dp);df_dp=tabla(res_dp,'keep_prob');mostrar(df_dp)
fig,ax=plt.subplots();plot_acc(df_dp,df_base,'keep probability','Stochastic dropout',logx=False,ax=ax);plt.show()

## 10. Data augmentation sweep
Synthetic samples are produced **only from training examples**. Gaussian jitter and within-class interpolation are exploratory; morphometric plausibility is not guaranteed. Standardized features are used. Report this limitation.

In [ ]:
cfg_aug=[dict(label=f'Jitter k={k} σ={sigma}',reg='none',aug='jitter',k=k,sigma=sigma) for k in (1,3) for sigma in (.02,.1)]
cfg_aug += [dict(label=f'Interpolation k={k}',reg='none',aug='interp',k=k) for k in (1,3)]
res_aug=run_sweep(cfg_aug);df_aug=tabla(res_aug);mostrar(df_aug)

## 11. Comparison and held-out test
Select each method's best configuration using **validation only**. Then compare the chosen methods. To avoid optimistic selection, choose one overall winner by validation and evaluate it once on the held-out test set.

In [ ]:
all_groups={'Baseline':res_base,'L2':res_l2,'L1':res_l1,'Dropout':res_dp,'Augmentation':res_aug}
selection=[]
for method,results in all_groups.items():
    summary=tabla(results)
    row=summary.sort_values('val_%',ascending=False).iloc[0]
    selection.append(dict(method=method,configuration=row['configuración'],train_pct=row['train_%'],val_pct=row['val_%'],val_std=row['val_std']))
comparison=pd.DataFrame(selection).sort_values('val_pct',ascending=False)
display(comparison.round(3))
comparison.to_csv(PROJECT/'results'/'method_comparison.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4));ax.bar(comparison.method,comparison.val_pct);ax.set_ylabel('Validation accuracy (%)');ax.set_title('Best configuration per method');plt.xticks(rotation=20);plt.tight_layout();plt.show()

In [ ]:
winner=comparison.iloc[0]
all_results=sum(all_groups.values(),[])
chosen=next(r['cfg'] for r in all_results if r['label']==winner['configuration'])
# Refit selected configuration on TRAIN ONLY (same seeds); test set never used for tuning.
test_rows=[]
for seed in SEEDS:
    params,metrics=fit(chosen,seed)
    test_rows.append(dict(seed=seed,train_acc=metrics['train_acc'],val_acc=metrics['val_acc'],test_acc=accuracy(test_X,test_Y,params)))
final_test=pd.DataFrame(test_rows)
display(final_test)
print('Selected:',winner['method'],winner['configuration'])
print('Held-out test accuracy mean ± std: %.3f ± %.3f' % (final_test.test_acc.mean(),final_test.test_acc.std(ddof=0)))
final_test.to_csv(PROJECT/'results'/'heldout_test.csv',index=False)

## 12. Discussion — complete after executing
1. Which λ improved validation accuracy? At which λ did underfitting emerge?
2. Compare L1 vs L2 weight magnitudes and the fraction of near-zero weights (not necessarily exact zeros).
3. Did stochastic dropout help on this small tabular dataset?
4. Did augmentation improve validation? Discuss possible implausible synthetic morphologies.
5. Report the selected model's **held-out test** accuracy, including variability across seeds.
6. Limitations: only 900 samples, 3 seeds by default, synthetic-feature plausibility, and multiple hyperparameter comparisons.

**Submission:** Download the `.ipynb` as a file for Canvas; do not submit only a Colab link. The PDF report is a separate deliverable.